In [ ]:
# Skeleton Training Prototype
# ===========================
#
# - This is the jupyter notebook I created to define and test out the training process for Skeleton -> moved to google colab
#     - a cross-attention transformer that predicts the next coordinate (3D vector) relative to a ligand centroid given its molecular fingerprint
# - You can run this notebooks's cells yourself, but just be aware the model that gets produced is watered down and its almost definitely going to output coordinates that oscillate around the same area.
#     - Compared to the google colab file, it has half the embedding size, much lower batch size, less heads and extremely restricted training data (compared to the total 14M+)
#     - still fun to mess around with though, I was able to get some prototypes going that could kind of grasp the general pattern
#     - also included some small tests on it to see if it was learning anything (checking output for different molecules, coordinate output statistics)

In [1]:
import torch
import pandas as pd
%matplotlib inline
from pathlib import Path

In [2]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

data_folder = Path
RADIAL_SEQUENCES = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
MOLECULAR_FINGERPRINTS = pd.read_pickle(DATAPATH / "molfp_df.pkl")
TRAIN_POINTERS = pd.read_parquet(DATAPATH / "training_pointers.parquet")
TEST_POINTERS = pd.read_parquet(DATAPATH / "test_pointers.parquet")

In [3]:
print(MOLECULAR_FINGERPRINTS.head(3))
print(RADIAL_SEQUENCES.head(3))

sample_fp = MOLECULAR_FINGERPRINTS["map4_fp"].iloc[0]
print(len(sample_fp), sample_fp[0].dtype)

                                          smiles_str  \
0  C[C@@H](N1CC[C@](C)(C1=O)c1ccc(OCc2cc(C)nc3ccc...   
1  Cc1ccc(CNc2cc(OC[C@H]3C[C@@H]3c3ccc4CCCc4n3)nn...   
2  C[C@@H](COc1ccnc2CCC[C@@H](C)c12)C[C@H]1Cc2cc3...   

                                             map4_fp  
0  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
1  [tensor(0.), tensor(1.), tensor(0.), tensor(0....  
2  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
  PDB_ID                                    radial_sequence
0   5UEY  [[[I], [62, 57, 33]], [[K], [69, 54, 42]], [[M...
1   6FID  [[[Y], [47, 68, 41]], [[D], [68, 60, 53]], [[H...
2   1WCC  [[[L], [36, 48, 37]], [[K], [66, 62, 45]], [[V...
1024 torch.float32


In [4]:
training_pointer = TRAIN_POINTERS.sample(10000)
test_pointer = TEST_POINTERS.sample(3000)

In [5]:
from torch.utils.data import Dataset

class RadialDataset(Dataset):

    def __init__(self, pointer, smiles_molfp, pdb_radial,
                 block_size, radial_resolution):
        self.pointer = pointer
        self.smiles_molfp = smiles_molfp.set_index("smiles_str")       # <class 'torch.Tensor'>
        self.pdb_radial = pdb_radial.set_index("PDB_ID")               # <class 'list'>
        # set index moves "column label" column to row label, enabling O(1) .loc["target_ID"] lookups

        self.block_size = block_size
        self.radial_resolution = radial_resolution


    def __len__(self):
        return len(self.pointer)


    def __getitem__(self, idx):
        row = self.pointer.iloc[idx]
        smiles = row["SMILES"]
        pdb_id = row["PDB_HIT"]
        target_idx = row["WINDOW_IDX"]

        # context query
        molecular_fingerprint = self.smiles_molfp.loc[smiles, "map4_fp"]

        # radial sequence query
        radial_sequence = self.pdb_radial.loc[pdb_id, "radial_sequence"]

        radial_X, radial_Y = self.radial_XY(radial_sequence, target_idx)

        return (molecular_fingerprint,
                torch.tensor(radial_X, dtype=torch.float32),     # block_size, 3
                torch.tensor(radial_Y, dtype=torch.float32))     # 3,


    def radial_XY(self, radial_seq, target_idx):
        """
        Generates X and Y set for a given radial sequence + molecular fingerprint in the background?
        """
        o_num = int(self.radial_resolution + 1)
        o_num = int(self.radial_resolution * 1.5)  # max range + 1
        context = [[o_num, o_num, o_num] for _ in range(self.block_size)]
        for idx in range(len(radial_seq)):
            coords = radial_seq[idx][1]  # = [X, Y, Z]
            if idx == target_idx:
                return context, coords  # This is our XY data
            context = context[1:]+[coords]  # sliding window append -> context style in nanogpt

In [6]:
# RadialSeeker parameters used -> record these - might design a config file later
radial_resolution = 100
intrashell_resolution = 100
max_angstroms = 33

block_size = 75
batch_size = 64
n_embd = 256
n_head = 4
n_layers = 4
map4_res = 1024
dropout=0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'
loss_alpha = 0.5
n_epochs = 5

# training dataset
training_dataset = RadialDataset(pointer=training_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 radial_resolution=radial_resolution)

# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

torch.manual_seed(311104)

# test / validation set
test_set = RadialDataset(pointer=test_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 radial_resolution=radial_resolution)
# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=4,
    shuffle=False,
    drop_last=True
)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

# SKELETON MODEL
class SkeletonModel(nn.Module):
    def __init__(self, n_embd, n_head, n_layers, block_size,
                 map4_res, radial_resolution,
                 l_a, dropout):
        super().__init__()
        self.block_size = block_size
        self.map4_res = map4_res
        self.radial_resolution = radial_resolution

        self.c_project = nn.Linear(3, n_embd)  # feed coordinates here
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.mol_encoder = nn.Sequential(
            nn.Linear(map4_res, int(map4_res//2)),
            nn.ReLU(),
            nn.Linear(int(map4_res//2), n_embd),
            nn.LayerNorm(n_embd)
        )


        self.stacks = nn.ModuleList([Stack(n_embd, n_head, block_size, dropout) for _ in range(n_layers)])

        # OUTPUT HEAD -> outputs coordinates
        self.ln_f = nn.LayerNorm(n_embd)
        self.c_head = nn.Linear(n_embd, 3)

        self.loss_alpha = l_a


    def forward(self, coord_hist, map4_enc, targets=None):
        """
        coord_hist will be [n_batch, block_size, 3]
        targets is [n_batch, 1, 3]
        map4_enc -> unsure how I'm going to encode this
        """
        B, T, C = coord_hist.shape
        coordinate_emb = self.c_project(coord_hist.float())
        pos_emb = self.pos_emb(torch.arange(T, device=coord_hist.device))
        x = coordinate_emb + pos_emb

        mol_emb = self.mol_encoder(map4_enc.float()).unsqueeze(1)

        for stack in self.stacks:
            x = stack(x, mol_emb)

        output_coords = torch.sigmoid(self.c_head(self.ln_f(x[:, -1, :]))) * (self.radial_resolution + 1)

        loss = None
        if targets is not None:
            targets = targets.float()
            mse_loss = F.mse_loss(output_coords, targets)
            euc_loss = torch.sqrt(((output_coords - targets)**2).sum(dim=-1) + 1e-8).mean()  # euclidean distance between the two

            loss = mse_loss * 0.5 + euc_loss * 0.5

        return output_coords, loss

    def generate(self, map4_enc, max_AAs=130):
        # largest protein pocket in dataset was 107
        map4_enc = map4_enc.to(next(self.parameters()).device)
        coord_context = torch.full((1, block_size, 3), float(self.radial_resolution * 1.5), device=map4_enc.device)
        coord_out = []

        for _ in range(max_AAs):
            next_coord, _ = self.forward(coord_context, map4_enc)
            coord_out.append(next_coord)
            coord_context = torch.cat([coord_context[:, 1:, :], next_coord.unsqueeze(1)], dim=1)

        return coord_out


class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

        # added in a causal mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute affinities
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5
        attn_wts = attn_wts.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighed aggregation
        values = self.value(x)
        output = attn_wts @ values
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coordinate_emb, molecular_emb):
        q = self.query(coordinate_emb)
        k = self.key(molecular_emb)
        v = self.value(molecular_emb)

        head_size = q.shape[-1]

        c_attn = q @ k.transpose(-2, -1) * head_size**-0.5
        c_attn = F.softmax(c_attn, dim=-1)
        c_attn = self.dropout(c_attn)

        return c_attn @ v

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,
                                                       head_size,
                                                       dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coord_emb, mol_emb):
        out = torch.cat([h(coord_emb, mol_emb) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class FeedForwardNet(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Stack(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.satt_head = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForwardNet(n_embd, dropout)
        self.catt_head = MultiHeadCrossAttention(n_embd, n_head, dropout)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, mol_emb):
        x = x + self.satt_head(self.ln1(x))
        x = x + self.catt_head(self.ln2(x), mol_emb)
        x = x + self.ffwd(self.ln3(x))
        return x

In [8]:
model = SkeletonModel(n_embd=n_embd, n_head=n_head, n_layers=n_layers,
                      block_size=block_size, map4_res=map4_res, radial_resolution=radial_resolution,
                      l_a=loss_alpha, dropout=dropout).to(device)

In [9]:
learning_rate = 3e-4   # maybe try 5e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

SkeletonModel(
  (c_project): Linear(in_features=3, out_features=256, bias=True)
  (pos_emb): Embedding(75, 256)
  (mol_encoder): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (stacks): ModuleList(
    (0-3): 4 x Stack(
      (satt_head): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=256, out_features=64, bias=False)
            (query): Linear(in_features=256, out_features=64, bias=False)
            (value): Linear(in_features=256, out_features=64, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (projection): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffwd): FeedForwardNet(
        (net): Sequential(
          (0): Linear(in_

In [10]:
@torch.no_grad()
def estimate_loss(model, loader, device, radial_res, max_batches=None):
    model.eval()
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches and batch_idx >= max_batches:
            break

        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        _, loss = model(radial_X, map4_fp, radial_Y)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches


In [ ]:
for epoch in range(n_epochs):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        optimizer.zero_grad()
        pred, loss = model(radial_X, map4_fp, radial_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    train_loss = total_loss / len(loader)
    val_loss = estimate_loss(model, test_loader, device, radial_res=radial_resolution, max_batches=batch_size)

    print(f"Epoch {epoch+1} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Gap: {val_loss - train_loss:.6f}")


Epoch 1 | Batch 1 | Loss: 180.249741


In [ ]:
from src.nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"
sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
model.generate(sample_map4)

In [ ]:
# grab a batch and look at target distribution
map4_fp, radial_X, radial_Y = next(iter(loader))
print(f"Target mean: {radial_Y.float().mean():.4f}")
print(f"Target std:  {radial_Y.float().std():.4f}")
print(f"Target min:  {radial_Y.float().min():.4f}")
print(f"Target max:  {radial_Y.float().max():.4f}")

In [ ]:
seq_lengths = []
for seq in RADIAL_SEQUENCES["radial_sequence"]:
    seq_lengths.append(len(seq))

seq_lengths = pd.Series(seq_lengths)
print(seq_lengths.describe())
print(f"Median: {seq_lengths.median()}")
print(f"90th percentile: {seq_lengths.quantile(0.90)}")
print(f"95th percentile: {seq_lengths.quantile(0.95)}")

In [ ]:
model.eval()
with torch.no_grad():
    ctx = torch.zeros(1, block_size, 3, device=device)

    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(ctx, fp)
        print(f"{name}: {out.squeeze().tolist()}")

In [ ]:
model.eval()
all_preds = []
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i >= 20: break
        map4_fp, radial_X, radial_Y = batch
        map4_fp, radial_X = map4_fp.to(device), radial_X.to(device)
        pred, _ = model(radial_X, map4_fp)
        all_preds.append(pred.cpu())

all_preds = torch.cat(all_preds)
print(f"Pred mean: {all_preds.mean():.4f}")
print(f"Pred std:  {all_preds.std():.4f}")
print(f"Pred min:  {all_preds.min():.4f}")
print(f"Pred max:  {all_preds.max():.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    fp = torch.tensor(smiles_fingerprint("CC(=O)Oc1ccccc1C(=O)O"),
                     dtype=torch.float32).unsqueeze(0).to(device)

    # try different context windows - all padding vs partial real context
    ctx_empty = torch.full((1, block_size, 3),
                           float(radial_resolution * 1.5), device=device)
    ctx_mid = torch.full((1, block_size, 3),
                         float(50), device=device)  # mid-sequence
    ctx_late = torch.full((1, block_size, 3),
                          float(10), device=device)  # near end

    for name, ctx in [("empty", ctx_empty), ("mid", ctx_mid), ("late", ctx_late)]:
        out, _ = model(ctx, fp)
        print(f"{name} context: {out.squeeze().tolist()}")